## Estadistica Apicada

Actividad 6 - 25 Septiembre

Bryan Alejandro Estrada Rodriguez 1844554

Grupo 42

## 1. Importa las librerías básicas y especializadas que consideres necesarias


In [3]:
# Librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
!pip install lifelines

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.7/350.7 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 10.5 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=f0d4a81d29cf0f1a7e9c6ca06ca4dfa3a867ba1fdeb320015df8505b4ed8fc2d
  Stored in directory: /root/.cache/pip/wheels/25/cc/e0/ef2969164144c899fedb22b338f6703e2b9cf46eeebf254991
Successfully built autograd-gamma


## 2. Importa la base de datos

 https://raw.githubusercontent.com/jimmyzac/Estadistica-Aplicada-FCFM-UANL/main/bases_datos/desercion_laboral.csv

In [22]:
# Librerias especializadas
from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullFitter, ExponentialFitter, LogNormalFitter

In [6]:
desercion_laboral = pd.read_csv('https://raw.githubusercontent.com/jimmyzac/Estadistica-Aplicada-FCFM-UANL/main/bases_datos/desercion_laboral.csv')
desercion_laboral


,satisfaccion,evaluacion,proyectos,promedio_horas_mes,semestre,accidente,renunciar,ascenso_5years,departamento,salario
0,3.8,5.3,2,157,3,0,1,0,ventas,bajo
1,8.0,8.6,5,262,6,0,1,0,ventas,medio
2,1.1,8.8,7,272,4,0,1,0,ventas,medio
3,7.2,8.7,5,223,5,0,1,0,ventas,bajo
4,3.7,5.2,2,159,3,0,1,0,ventas,bajo
...,...,...,...,...,...,...,...,...,...,...
14994,4.0,5.7,2,151,3,0,1,0,soporte,bajo
14995,3.7,4.8,2,160,3,0,1,0,soporte,bajo
14996,3.7,5.3,2,143,3,0,1,0,soporte,bajo
14997,1.1,9.6,6,280,4,0,1,0,soporte,bajo


## 3. Depurar la base de datos y estadísticas descriptivas

In [7]:
# Buscamos datos perdidos
desercion_laboral.isnull().sum()

satisfaccion           0
evaluacion             0
proyectos              0
promedio_horas_mes     0
semestre               0
accidente              0
renunciar              0
ascenso_5years         0
departamento          26
salario                0
dtype: int64

In [19]:
# Entontramos que si hay 26 missing values,
#por lo que optamos por eliminarlos ya que la base de datos es muy grande y nos hace pensar que no habra inconveniente con los resultados
desercion_laboral=desercion_laboral.dropna()

In [9]:
# Buscamos si hay duplicados en el DataFrame
desercion_laboral.duplicated().sum()

2982

In [10]:
#Eliminamos los datos duplicados
desercion_laboral=desercion_laboral.drop_duplicates(keep='first').reset_index(drop=True)


In [11]:
## Verificar que todas las variables son numericas
desercion_laboral.dtypes

satisfaccion          float64
evaluacion            float64
proyectos               int64
promedio_horas_mes      int64
semestre                int64
accidente               int64
renunciar               int64
ascenso_5years          int64
departamento           object
salario                object
dtype: object

In [12]:
desercion_laboral['departamento'].value_counts()

ventas                    3239
mantenimiento             2244
soporte                   1821
informática                976
Investigación              694
desarrollo de producto     686
mercadotecnia              673
contabilidad               621
recursos humanos           601
administración             436
Name: departamento, dtype: int64

In [14]:
# Convertimos las Variables que no son numericas
desercion_laboral = pd.get_dummies(desercion_laboral,drop_first=True)
desercion_laboral.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11991 entries, 0 to 11990
Data columns (total 19 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   satisfaccion                         11991 non-null  float64
 1   evaluacion                           11991 non-null  float64
 2   proyectos                            11991 non-null  int64  
 3   promedio_horas_mes                   11991 non-null  int64  
 4   semestre                             11991 non-null  int64  
 5   accidente                            11991 non-null  int64  
 6   renunciar                            11991 non-null  int64  
 7   ascenso_5years                       11991 non-null  int64  
 8   departamento_administración          11991 non-null  uint8  
 9   departamento_contabilidad            11991 non-null  uint8  
 10  departamento_desarrollo de producto  11991 non-null  uint8  
 11  departamento_informática    

In [15]:
# Estadisticas descriptivas
desercion_laboral.describe()

,satisfaccion,evaluacion,proyectos,promedio_horas_mes,semestre,accidente,renunciar,ascenso_5years,departamento_administración,departamento_contabilidad,departamento_desarrollo de producto,departamento_informática,departamento_mantenimiento,departamento_mercadotecnia,departamento_recursos humanos,departamento_soporte,departamento_ventas,salario_bajo,salario_medio
count,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.00000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000,11991.000000
mean,6.296581,7.166825,3.802852,200.473522,3.364857,0.154282,0.166041,0.016929,0.036361,0.051789,0.057210,0.081394,0.18714,0.056125,0.050121,0.151864,0.270119,0.478692,0.438746
std,2.410700,1.683426,1.163238,48.727813,1.330240,0.361234,0.372133,0.129012,0.187194,0.221610,0.232252,0.273451,0.39004,0.230173,0.218204,0.358904,0.444040,0.499567,0.496254
min,0.900000,3.600000,2.000000,96.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.800000,5.700000,3.000000,157.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.600000,7.200000,4.000000,200.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,8.200000,8.600000,5.000000,243.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
max,10.000000,10.000000,7.000000,310.000000,10.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 4. Estimar el modelode Cox e interpretar cada uno de los coeficientes

In [16]:
## Estimar el modelo
#cph1 = CoxPHFitter(variable,tiempo, evento)
cph1 = CoxPHFitter().fit(desercion_laboral,'semestre','renunciar')
cph1.print_summary()

<lifelines.CoxPHFitter: fitted with 11991 total observations, 10000 right-censored observations>
             duration col = 'semestre'
                event col = 'renunciar'
      baseline estimation = breslow
   number of observations = 11991
number of events observed = 1991
   partial log-likelihood = -15884.05
         time fit was run = 2023-09-27 05:19:40 UTC

---
                                      coef  exp(coef)   se(coef)   coef lower 95%   coef upper 95%  exp(coef) lower 95%  exp(coef) upper 95%
covariate                                                                                                                                   
satisfaccion                         -0.24       0.78       0.01            -0.26            -0.22                 0.77                 0.80
evaluacion                           -0.01       0.99       0.01            -0.04             0.02                 0.96                 1.02
proyectos                            -0.29       0.75       0.02            -0.33            -0.25                 0.72                 0.78
promedio_horas_mes                    0.00       1.00       0.00             0.00             0.00                 1.00                 1.00
accidente                            -1.25       0.29       0.10            -1.44            -1.05                 0.24                 0.35
ascenso_5years                       -1.38       0.25       0.36            -2.08            -0.68                 0.13                 0.51
departamento_administración           0.22       1.25       0.18            -0.13             0.57                 0.88                 1.76
departamento_contabilidad             0.24       1.27       0.14            -0.04             0.52                 0.96                 1.69
departamento_desarrollo de producto   0.36       1.43       0.14             0.08             0.64                 1.08                 1.90
departamento_informática              0.29       1.34       0.13             0.03             0.56                 1.03                 1.74
departamento_mantenimiento            0.43       1.54       0.12             0.20             0.67                 1.22                 1.95
departamento_mercadotecnia            0.34       1.40       0.14             0.06             0.62                 1.06                 1.86
departamento_recursos humanos         0.43       1.54       0.14             0.15             0.72                 1.16                 2.05
departamento_soporte                  0.45       1.57       0.12             0.21             0.69                 1.23                 2.00
departamento_ventas                   0.35       1.42       0.12             0.12             0.58                 1.13                 1.78
salario_bajo                          1.54       4.66       0.15             1.25             1.83                 3.48                 6.23
salario_medio                         1.15       3.15       0.15             0.85             1.44                 2.35                 4.22

                                      cmp to      z      p   -log2(p)
covariate                                                            
satisfaccion                            0.00 -26.89 <0.005     526.58
evaluacion                              0.00  -0.76   0.44       1.17
proyectos                               0.00 -14.75 <0.005     161.15
promedio_horas_mes                      0.00   3.88 <0.005      13.25
accidente                               0.00 -12.41 <0.005     115.04
ascenso_5years                          0.00  -3.88 <0.005      13.21
departamento_administración             0.00   1.24   0.21       2.22
departamento_contabilidad               0.00   1.66   0.10       3.36
departamento_desarrollo de producto     0.00   2.49   0.01       6.29
departamento_informática                0.00   2.17   0.03       5.06
departamento_mantenimiento              0.00   3.60 <0.005      11.64
departamento_mercadotecnia          

In [18]:
#  Nos muestra el exp(coef)
cph1.hazard_ratios_

covariate
satisfaccion                           0.784792
evaluacion                             0.989370
proyectos                              0.747063
promedio_horas_mes                     1.001841
accidente                              0.287456
ascenso_5years                         0.251557
departamento_administración            1.246432
departamento_contabilidad              1.271743
departamento_desarrollo de producto    1.433351
departamento_informática               1.339208
departamento_mantenimiento             1.540290
departamento_mercadotecnia             1.403331
departamento_recursos humanos          1.543940
departamento_soporte                   1.570015
departamento_ventas                    1.419844
salario_bajo                           4.659468
salario_medio                          3.150662
Name: exp(coef), dtype: float64

## Interpretar cada uno de los coeficientes

## Satisfaccion
Rechazamos H0, que tan satisfecho se encuentra la persona en el trabajo si influye en el riesgo de renunciar, disminuye un 22% por cada punto de satisfacción.

##Evaluació
No se rechaza H0, la evaluación que se le hace a los empleados no influye sobre el riesgo de que la persona renuncie.

##Proyectos
Rechazamos H0, el número de proyectos asignados al empleado si influye en el riesgo de que el empleado renuncie, este disminuye un 25% el riesgo por cada proyecto que se le asigne al empleado.

##Promedio_horas_mes
Rechazamos H0, el promedio de las horas trabajadas por mes aumenta el riesgo de que la persona renuncie en un 0.18% por cada hora.

##Accidente
Rechazamos H0, el haber tenido un accidente en la empresa si influye en el riesgo de que el empleado renuncie y se disminuye un 71% por cada accidente que tenga a comparación de las personas que no han tenido ningun accidente dentro de la empresa.

##Acenso_5years
Rechazamos H0, por lo que si es ascendido durante los últimos 5 años, disminuye el riesgo de que la persona renuncie en un 75% por cada ascenso durante los 5 años a comparación de las personas que no han tenido un ascenso durante los últimos 5 años.

##Departamento administración
NO rechazamos H0, el pertenecer al departamento de administración no influye en el riesgo de que el empleado renuncie.

##Departamento contabilidad
NO rechazamos H0, el pertenecer al departamento de contabilidad no influye en el riesgo de que el empleado renuncie.

##Departamento desarrollo de producto
Se rechaza H0, el pertenecer al departamento de desarrollo de producto influye en el riesgo de que el empleado renuncie y tiene 43% más riesgo que el departamento de investigación.

##Departamento informática
Rechazamos H0, el pertenecer al departamento de informática influye en el riesgo de que el empleado renuncie y tiene 34% más riesgo que el departamento de investigación.

##Departamento mantenimiento
Rechazamos H0, el pertenecer al departamento de mantenimiento influye en el riesgo de que el empleado renuncie y tiene 54% más riesgo que el departamento de investigación.

##Departamento mercadotecnia
Rechazamos H0, el pertenecer al departamento de mantenimiento influye en el riesgo de que el empleado renuncie y tiene 40% más riesgo que el departamento de investigación.

##Departamento recursos humanos
Rechazamos H0, el pertenecer al departamento de recursos humanos influye en el riesgo de que el empleado renuncie y tiene 54% más riesgo que el departamento de investigación.

##Departamento soporte
Rechazamos H0, el pertenecer al departamento de soporte influye en el riesgo de que el empleado renuncie y tiene 57% más riesgo que el departamento de investigación.

##Departamento ventas
Rechazamos H0, el pertenecer al departamento de ventas influye en el riesgo de que el empleado renuncie y tiene 42% más riesgo que el departamento de investigación.

##Salario bajo
Rechazamos H0; el tener un salario bajo influye sobre el riesgo, aumenta el riesgo un 366% más que el salario alto.

##Salario medio
Rechazamos H0; el tener un salario medio influye sobre el riesgo, aumenta el riesgo un 215% más que el salario alto.

## 5. Con base  en la interpretación de los coeficientes,¿qué recomendación podría hacerle  a  la  empresa  para  que  disminuya  la  cantidad  de  renuncias  de  los empleados?

Cuando un empleado se siente agusto trabajando para una empresa la probabilidad de que renuncie en poca, por lo que la satisfacción del empleado es un punto importante a considerar. Tomando el punto anterior, tambien sabemos que, mientras mas bajo sea el sueldo de un trabajador, aumenta la probabilidad de que renuncie en comparacion con los otros trabajadores que tienen un sueldo mas alto.


## 6. Pruebe si el modelo de Cox cumple el supuesto de riesgosproporcionales, de no hacerlo estime el modelo AFT adecuado

$H0$: Se cumple riesgo proporcional constante

$H1$: No Se cumple riesgo proporcional constante

In [20]:
cph1.check_assumptions(desercion_laboral,p_value_threshold=0.05)

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.



<lifelines.StatisticalResult: proportional_hazard_test>
 null_distribution = chi squared
degrees_of_freedom = 1
             model = <lifelines.CoxPHFitter: fitted with 11991 total observations, 10000 right-censored observations>
         test_name = proportional_hazard_test

---
                                          test_statistic      p  -log2(p)
accidente                           km              0.03   0.87      0.21
                                    rank            1.14   0.29      1.81
ascenso_5years                      km              3.66   0.06      4.17
                                    rank            6.71   0.01      6.71
departamento_administración         km              5.08   0.02      5.37
                                    rank            5.74   0.02      5.92
departamento_contabilidad           km              0.09   0.77      0.39
                                    rank            1.14   0.29      1.81
departamento_desarrollo de producto km              1.03   0.31      1.69
                                    rank            0.85   0.36      1.49
departamento_informática            km              0.48   0.49      1.03
                                    rank            0.53   0.47      1.10
departamento_mantenimiento          km              0.47   0.49      1.02
                                    rank            2.26   0.13      2.91
departamento_mercadotecnia          km              2.66   0.10      3.28
                                    rank            6.33   0.01      6.39
departamento_recursos humanos       km              0.68   0.41      1.29
                                    rank            2.86   0.09      3.46
departamento_soporte                km              0.00   0.98      0.03
                                    rank            0.89   0.35      1.53
departamento_ventas                 km              1.04   0.31      1.70
                                    rank            4.36   0.04      4.76
evaluacion                          km            443.05 <0.005    324.32
                                    rank          427.15 <0.005    312.82
promedio_horas_mes                  km            270.84 <0.005    199.74
                                    rank          307.41 <0.005    226.21
proyectos                           km            574.13 <0.005    419.05
                                    rank          684.93 <0.005    499.11
salario_bajo                        km              0.37   0.54      0.88
                                    rank            0.00   0.96      0.06
salario_medio                       km              0.16   0.69      0.53
                                    rank            0.10   0.75      0.42
satisfaccion                        km           1257.74 <0.005    912.74
                                    rank          845.24 <0.005    614.90



1. Variable 'satisfaccion' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'satisfaccion' might be incorrect. That is, there
may be non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'satisfaccion' using pd.cut, and then specify it in
`strata=['satisfaccion', ...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'evaluacion' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'evaluacion' might be incorrect. That is, there may
be non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in link [D] below on how to 

[]

Las variables satisfaccion, evaluacion, proyectos, promedio_hora_mes, ascenso_5years, departamento_administración, departamento_mercadotecnia y departamento_ventas no cumplen con el supuesto se estimará por el modelo de riesgo acelerado pero primero debemos encontrar el modelo adecuado

In [23]:
expo= ExponentialFitter().fit(desercion_laboral['semestre'],desercion_laboral['renunciar'])
weibull= WeibullFitter().fit(desercion_laboral['semestre'],desercion_laboral['renunciar'])
logn= LogNormalFitter().fit(desercion_laboral['semestre'],desercion_laboral['renunciar'])

(expo.AIC_,weibull.AIC_,logn.AIC_)

(15965.45892201228, 12511.512867661068, 11587.92169210356)

In [24]:
# El mejor modelo es Log Normal
from lifelines import LogNormalAFTFitter
logAFT=LogNormalAFTFitter().fit(desercion_laboral,'semestre','renunciar')
logAFT.print_summary()

<lifelines.LogNormalAFTFitter: fitted with 11991 total observations, 10000 right-censored observations>
             duration col = 'semestre'
                event col = 'renunciar'
   number of observations = 11991
number of events observed = 1991
           log-likelihood = -4876.57
         time fit was run = 2023-09-27 05:54:46 UTC

---
                                             coef  exp(coef)   se(coef)   coef lower 95%   coef upper 95%  exp(coef) lower 95%  exp(coef) upper 95%
param  covariate                                                                                                                                   
mu_    accidente                             0.26       1.30       0.02             0.22             0.30                 1.25                 1.35
       ascenso_5years                        0.37       1.44       0.06             0.24             0.49                 1.27                 1.63
       departamento_administración           0.05       1.05       0.04            -0.03             0.13                 0.97                 1.13
       departamento_contabilidad            -0.04       0.96       0.03            -0.11             0.02                 0.90                 1.02
       departamento_desarrollo de producto  -0.05       0.95       0.03            -0.12             0.01                 0.89                 1.01
       departamento_informática             -0.05       0.95       0.03            -0.11             0.01                 0.90                 1.02
       departamento_mantenimiento           -0.08       0.93       0.03            -0.13            -0.02                 0.88                 0.98
       departamento_mercadotecnia           -0.04       0.97       0.03            -0.10             0.03                 0.90                 1.03
       departamento_recursos humanos        -0.09       0.92       0.03            -0.15            -0.02                 0.86                 0.98
       departamento_soporte                 -0.08       0.92       0.03            -0.14            -0.03                 0.87                 0.97
       departamento_ventas                  -0.06       0.94       0.03            -0.11            -0.00                 0.90                 1.00
       evaluacion                            0.01       1.01       0.00            -0.00             0.01                 1.00                 1.01
       promedio_horas_mes                   -0.00       1.00       0.00            -0.00            -0.00                 1.00                 1.00
       proyectos                             0.10       1.10       0.01             0.09             0.11                 1.09                 1.11
       salario_bajo                         -0.37       0.69       0.03            -0.43            -0.31                 0.65                 0.73
       salario_medio                        -0.29       0.75       0.03            -0.34            -0.23                 0.71                 0.80
       satisfaccion                          0.07       1.07       0.00             0.06             0.07                 1.06                 1.07
       Intercept                             1.34       3.83       0.04             1.25             1.43                 3.50                 4.18
sigma_ Intercept                            -1.11       0.33       0.02            -1.14            -1.08                 0.32                 0.34

                                             cmp to      z      p   -log2(p)
param  covariate                                                            
mu_    accidente                               0.00  12.98 <0.005     125.59
       ascenso_5years                          0.00   5.80 <0.005      27.19
       departamento_administración             0.00   1.21   0.23       2.14
       departamento_contabilidad               0.00  -1.27   0.20       2.30
       departamento_desarrollo de producto     0.00  -1.54   0.12       3.01
       d

In [25]:
logAFT.params_

param   covariate                          
mu_     accidente                              0.263048
        ascenso_5years                         0.365029
        departamento_administración            0.047968
        departamento_contabilidad             -0.043292
        departamento_desarrollo de producto   -0.051817
        departamento_informática              -0.046400
        departamento_mantenimiento            -0.077644
        departamento_mercadotecnia            -0.035578
        departamento_recursos humanos         -0.086226
        departamento_soporte                  -0.083025
        departamento_ventas                   -0.057362
        evaluacion                             0.005721
        promedio_horas_mes                    -0.000309
        proyectos                              0.096128
        salario_bajo                          -0.369968
        salario_medio                         -0.286107
        satisfaccion                           0.067044
    

In [26]:
from math import exp
# exp(coef) de promedio_horas_mes
1-exp(-0.000309)

0.00030895226441685075